# 04: Implicit ALS Matrix Factorization vs TruncatedSVD Baseline
This notebook compares Implicit Feedback ALS (with confidence scaling $C_{ui} = 1 + \alpha r_{ui}$) against the classical TruncatedSVD baseline.

In [1]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np

# Robust root resolver whether run from notebooks/ or workspace root
_curr = Path('.').resolve()
PROJECT_ROOT = _curr if (_curr / 'src').exists() else (_curr.parent if (_curr.parent / 'src').exists() else (_curr.parents[1] if len(_curr.parents) > 1 and (_curr.parents[1] / 'src').exists() else _curr))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Switch working directory to project root so all relative paths ('data/...', 'artifacts/...') resolve seamlessly
os.chdir(PROJECT_ROOT)

from src.features.interaction_features import InteractionMatrixBuilder
from src.recommenders.als import ALSRecommender
from src.recommenders.svd import SVDRecommender


## 1. Load Matrix & Train ALS Model

In [2]:
train_df = pd.read_parquet('data/processed/train_events.parquet')
id_builder = InteractionMatrixBuilder().fit_mappings(train_df)
csr_mat, _ = id_builder.build_matrix(train_df)

als = ALSRecommender(factors=64, regularization=0.05, iterations=20, alpha_confidence=40.0)
als.fit(csr_mat, id_builder)
print("ALS Model Fitted.")


2026-10-09 07:27:49 [INFO] [interaction_features:50] Built ID mappings: 2996 unique users, 800 unique items.
2026-10-09 07:27:49 [INFO] [interaction_features:55] Computing weighted implicit interactions and building CSR matrix...
2026-10-09 07:27:49 [INFO] [interaction_features:87] Interaction matrix assembled: (2996, 800) with 30,869 non-zero entries.
2026-10-09 07:27:49 [INFO] [als_recommender:52] Training Implicit ALS (factors=64, reg=0.05, iter=20, alpha=40.0)...
2026-10-09 07:27:50 [INFO] [als_recommender:75] ALS model training complete.
ALS Model Fitted.


c:\Users\hp\AppData\Local\Programs\Python\Python311\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


## 2. Train TruncatedSVD Baseline

In [3]:
svd = SVDRecommender(n_components=32)
svd.fit(csr_mat, id_builder)
print(f"TruncatedSVD Fitted. Explained Variance Sum: {svd.svd.explained_variance_ratio_.sum():.4f}")


2026-10-09 07:27:55 [INFO] [svd_recommender:46] Training TruncatedSVD baseline with n_components=32...
2026-10-09 07:27:55 [INFO] [svd_recommender:64] TruncatedSVD baseline trained (explained variance ratio sum: 0.2807).
TruncatedSVD Fitted. Explained Variance Sum: 0.2807


## 3. Compare Personalization Recommendations

In [4]:
sample_user = list(id_builder.user_to_idx.keys())[0]
als_recs = als.recommend(sample_user, n=5)
svd_recs = svd.recommend(sample_user, n=5)

print(f"ALS Top-5 for Customer #{sample_user}: {als_recs}")
print(f"SVD Top-5 for Customer #{sample_user}: {svd_recs}")


ALS Top-5 for Customer #20000: [(1005, 1.0), (1000, 0.9353714454674658), (1055, 0.9321975682723049), (1609, 0.9277422593753545), (1002, 0.9047844657783145)]
SVD Top-5 for Customer #20000: [(1016, 1.0), (1033, 0.7678717970848083), (1504, 0.3027110695838928), (1113, 0.009969170205295086), (1433, 0.0)]
